# pip

In [ ]:
# !pip install pyarrow fastparquet
# !pip install plotly nbformat
# !pip install dash

# import

In [ ]:
import pandas as pd

import plotly.express as px

import plotly.graph_objects as go

from geopy.distance import geodesic

# Utils

In [ ]:
class Utils:
    DATA_file = "cd_142_dataset/raw_dataset"

    LIST_FILE_NAME = [
        DATA_file + r"/2014-03-12_vektory.csv",
        DATA_file + r"/2014-03-13_vektory.csv",
        DATA_file + r"/2014-03-14_vektory.csv",
        DATA_file + r"/2014-03-15_vektory.csv",
        DATA_file + r"/2014-03-16_vektory.csv",
        DATA_file + r"/2014-03-17_vektory.csv",
        DATA_file + r"/2014-03-18_vektory.csv",
        DATA_file + r"/2014-03-19_vektory.csv",
        DATA_file + r"/2014-03-20_vektory.csv",
        DATA_file + r"/2014-03-21_vektory.csv",
        DATA_file + r"/2014-03-22_vektory.csv",
        DATA_file + r"/2014-03-23_vektory.csv",
        DATA_file + r"/2014-03-24_vektory.csv",
        DATA_file + r"/2014-03-25_vektory.csv",
        DATA_file + r"/2014-03-26_vektory.csv",
    ]

In [ ]:
df_map = pd.read_csv('cells_142.csv', sep=',', encoding='utf-8')[['cellid', 'lat', 'lon']]

In [ ]:
def to_int(x):
    x = x.strip().replace('"', '')
    return int(x) if x.strip() != "" else None

In [ ]:
def convert_lat_lon_distance_to_meter(
    point1: tuple[float, float], point2: tuple[float, float]
) -> float:
    return geodesic(point1, point2).meters

In [ ]:
def distance_users_metric(trajectory1: pd.DataFrame, trajectory2: pd.DataFrame) -> float:
    total_distance = 0.0
    count = 0

    for _, row1 in trajectory1.iterrows():
        for _, row2 in trajectory2.iterrows():
            point1 = (row1["latitude"], row1["longitude"])
            point2 = (row2["latitude"], row2["longitude"])
            distance = convert_lat_lon_distance_to_meter(point1, point2)
            total_distance += distance
            count += 1

    return total_distance / count if count > 0 else float("inf")

In [ ]:
import numpy as np
import app.utils as utils

dist_matrix = np.load(utils.PATH_SOURCE / "dist_matrix.npy", allow_pickle=True)
cell_ids = np.load(utils.PATH_SOURCE / "cell_ids.npy", allow_pickle=True)

index_map = {cell_id: i for i, cell_id in enumerate(cell_ids)}

# Analyse

In [ ]:
data = []

with open(Utils.LIST_FILE_NAME[0], encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(";")

        # Vecteur Standard
        meta = {
            "id": to_int(parts[0]),
            "age": to_int(parts[1]),
            "sexe": parts[2],
            "Big_number": to_int(parts[3]),
            "comportement": parts[4],
            "cellule_home": parts[5],
            "cellule_work": parts[6],
            "nb_records": to_int(parts[7])
        }

        # Vecteur records
        events_raw = parts[8:]

        events = []
        for i in range(0, len(events_raw), 2):
            if i + 1 < len(events_raw):
                code = events_raw[i]
                s = events_raw[i + 1]
                events.append((code, int(s)))

        data.append({
            "meta": meta,
            "events": events
        })

In [ ]:
df_vector_standard = pd.DataFrame([d["meta"] for d in data])
df_vector_standard

In [ ]:
data

# Map

#### Trajet d'un utilisateur

In [ ]:
user_id = 1769

user = next(d for d in data if d["meta"]["id"] == user_id)
events = user["events"]

In [ ]:
df_map = pd.read_csv('cells_142.csv', sep=',', encoding='utf-8')[['cellid', 'lat', 'lon']]

In [ ]:
coord_map = {
    row["cellid"]: (row["lat"], row["lon"])
    for _, row in df_map.iterrows()
}

In [ ]:
trajectory = []

for cell, t  in events:
    if cell in coord_map:
        trajectory.append({
            "time": t,
            "latitude": coord_map[cell][0],
            "longitude": coord_map[cell][1]
        })

# Preprocessing

# Raw database

In [ ]:
import pyarrow.parquet as pq

table = pq.read_table(
    "raw_parquet/part-r-00002-a6ae3bcf-6897-472a-a7d4-4be40e4ac58b.gz.parquet"
)

df = table.to_pandas(types_mapper=None)

In [ ]:
df